In [17]:
import gymnasium as gym
from model import Policy
import torch
import numpy as np

In [ ]:
# IN RL 
# 1) we see the environment (observation)
# 2) Agent takes an action based on observation
# 3) Agent gets some form of reward and new state
# 4) Agent updates policy with reward.
# Rinse and Repeat

# Create the Env and reset it
env_id = "CartPole-v1"
env = gym.make(env_id)
state, info = env.reset()

# Create the policy
LR = 1e-5
GAMMA = 0.99
model = Policy()
optimizer = torch.optim.AdamW(model.parameters(), lr = LR)

# Environment HP
EPISODES = 10000
episode_steps = 100
EPISODE_SCORES = [] # Total sum of rewards

for eps in range(EPISODES):
    # Reset the environment.
    observation, info = env.reset()

    _eps_log_probs = []
    _eps_rewards = []

    for time_step in range(episode_steps):
        # Policy makes the decision.
        action, log_probs = model.act(state)

        state, reward, terminated, truncated, info = env.step(action)

        # Game ended earlier
        if terminated or truncated:
            break

        _eps_rewards.append(reward)
        _eps_log_probs.append(log_probs)

    EPISODE_SCORES.append(np.sum(_eps_rewards))
    # let's ignore reward to go policy for now.
    # Now to calculate policy gradient.

    # Reorders this into
    #r_t, r_t-1, r_t-2, ... , r0
    returns = []
    for r in _eps_rewards[::-1]:
        disc_return = returns[-1] if len(returns) > 0 else 0
        returns.append(GAMMA * disc_return + r)

    assert len(returns) == len(_eps_log_probs)

    log_loss_rewards = []
    for reward_to_go, log_probs in zip(returns, _eps_log_probs):
        log_loss_rewards.append(-torch.tensor(reward_to_go) * log_probs)

    #create one tensor and sum it 
    policy_loss = torch.cat(log_loss_rewards).sum() 
    optimizer.zero_grad()
    policy_loss.backward() # compute_gradients
    optimizer.step() # update weights

    # Scores are going up!!!
    if eps % 100 == 0:
        print(f"Episode mean score: {np.mean(EPISODE_SCORES)}") 

Episode mean score: 17.0
Episode mean score: 20.03960396039604
Episode mean score: 19.970149253731343
Episode mean score: 20.146179401993354
Episode mean score: 20.635910224438902
Episode mean score: 20.760479041916167
Episode mean score: 20.778702163061563
Episode mean score: 20.647646219686163
Episode mean score: 20.860174781523096
Episode mean score: 20.751387347391788
Episode mean score: 20.555444555444556
Episode mean score: 20.627611262488646


In [ ]:
torch.tensor([1])